# The Data Science Process in Pure Python
### Case study: *How proficient is our class in Python?*

**Programming for Data Science**

In this notebook we walk through a complete (small) data science process using **plain Python only**: no NumPy, no pandas, no matplotlib. Doing it by hand once shows you exactly what those libraries will do for you later, and gives you practice with the core Python skills every data programmer needs.

```
 Ask a question → Collect data → Explore → Preprocess → Verify → Explore again
        ↑                                                            │
        └──────────────── Refine the question ←──────────────────────┘
                                   ↓
                         Analyse → Communicate
```

**Rules for this notebook**

| Allowed | Not yet |
|---|---|
| Built-in Python modules `csv`, `datetime`, `math`, `collections` | `numpy`, `pandas`, `matplotlib`, `statistics`* |

\* `statistics` is used **only to check** the functions you write yourself, never to compute the answers.

**Programming principles you will see along the way** (look for the 💡 boxes):
abstraction, DRY, readability, defensive code, verification, testing, reproducibility.

## Setup

In [1]:
import io
import csv
import datetime as dt
import math
from collections import Counter

import statistics  # ONLY for checking our own functions with assert

DATA_PATH = "Data/PythonQuiz_CQ_2026_24.csv"   # relative path: works on every machine

>**Principle: Reproducibility.** All imports go in the first code cell, and the data file is referenced by a **relative** path stored in **one** constant. Anyone who clones the course folder can run this notebook unchanged. Before submitting, always run *Kernel → Restart & Run All*.

## 1. Ask a question

> **How proficient is the class in Python?**

This question is still vague. That's normal: we will make it precise *after* we have looked at the data (step 7).

## 2. Collect data

Students took a Python quiz on **Moodle**. Moodle can export the results as a `.csv` file

- One row per **attempt**, plus one summary row at the end (the class average)
- Names, student IDs and e-mails were removed before sharing the file

> ⚠️ **Data ethics.** Grades are personal data. Always remove identifying columns before sharing or publishing a dataset.

## 3. Load the data

### 3.1 Why not just `line.split(",")`?

Splitting each line on commas works on *this* file, but it breaks as soon as a value contains a comma. CSV files protect such values with quotes, and the `csv` module understands the quotes:

In [2]:
line = 'Finished,"Nguyen, Van A",8.5'
print(line.split(","))                        # 4 values: wrong
print(next(csv.reader(io.StringIO(line))))    # 3 values: right

['Finished', '"Nguyen', ' Van A"', '8.5']
['Finished', 'Nguyen, Van A', '8.5']


In [3]:
line = 'Finished,"Nguyen, Van A",8.5'
print(next(csv.reader(io.StringIO(line))))    # 3 values: right

['Finished', 'Nguyen, Van A', '8.5']


### 3.2 A function to load any CSV file

We store the table **column by column**: a dictionary that maps each column name to the list of its values. (pandas stores data column by column too, which is why this will feel familiar in Topic 7.)

In [4]:
def load_csv(path):
    """Read a CSV file into a dict that maps each column name to a list of strings.

    Raises ValueError if a row has a different number of values than the header.
    """
    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        columns = {name: [] for name in header}
        for row in reader:
            if len(row) != len(header):
                raise ValueError(
                    f"{path}, line {reader.line_num}: "
                    f"expected {len(header)} values, got {len(row)}"
                )
            for name, value in zip(header, row):
                columns[name].append(value)
    return columns


raw = load_csv(DATA_PATH)
list(raw)          # the column names

['Status',
 'Started',
 'Completed',
 'Duration',
 'Grade/10.00',
 'Q. 1 /0.53',
 'Q. 2 /0.53',
 'Q. 3 /0.53',
 'Q. 4 /0.53',
 'Q. 5 /0.53',
 'Q. 6 /0.53',
 'Q. 7 /0.53',
 'Q. 8 /0.53',
 'Q. 9 /0.53',
 'Q. 10 /0.53',
 'Q. 11 /0.53',
 'Q. 12 /0.53',
 'Q. 13 /0.53',
 'Q. 14 /0.53',
 'Q. 15 /0.53',
 'Q. 16 /0.53',
 'Q. 17 /0.53',
 'Q. 18 /0.53',
 'Q. 19 /0.53']

`encoding="utf-8-sig"`: Moodle (like Excel) writes an invisible **BOM** character at the start of the file. With plain `"utf-8"`, the first column name would be `'\ufeffState'` instead of `'State'`.

>**Principle: Abstraction.** `load_csv` hides *how* a CSV file is read behind a name that says *what* it does. It has one job, takes its input as a parameter, `return`s its result, and fails **loudly** (with file and line number) instead of silently loading a broken row.

We will need two more small helpers again and again:

In [5]:
def num_rows(columns):
    """Number of rows in a column dict."""
    return len(next(iter(columns.values())))
    # or len(columns['Status'])


def get_row(columns, i):
    """Return row i as a dict {column name: value}."""
    return {name: values[i] for name, values in columns.items()}

## 4. Explore the raw data

### 4.1 How many rows and columns?

In [6]:
print("rows:   ", num_rows(raw))
print("columns:", len(raw))

rows:    28
columns: 24


### 4.2 What does a row mean? Do all rows mean the same thing?

In [7]:
Counter(raw["Status"])

Counter({'Finished': 27, '': 1})

In [8]:
for i in range(num_rows(raw)):
    if raw["Status"][i] != "Finished":
        row = get_row(raw, i)
        print(i, row["Status"] or "(empty)", row["Started"] or "(empty)",
              row["Grade/10.00"], row["Q. 1 /0.53"])

27 (empty) (empty) 7.31 0.37


There are **three kinds of rows**:

| `State` | Meaning | Keep? |
|---|---|---|
| `Finished` | the student clicked *Submit all and finish* | ✅ |
| `In progress` | the attempt was never submitted; every score is `-` | ❌ not a real result |
| *(empty)* | the last row: Moodle's **class average** for each column | ❌ not a student at all |

So a row is **one quiz attempt**, but only the `Finished` rows are the attempts we want to analyse.

### 4.3 Are there duplicated rows?

In [9]:
row_counts = Counter(tuple(get_row(raw, i).values()) for i in range(num_rows(raw)))
[row for row, n in row_counts.items() if n > 1]

[]

No row appears twice.

### 4.4 What does each column mean?

| Column | Meaning |
|---|---|
| `Status` | status of the attempt (`Finished`, `In progress`) |
| `Started`, `Completed` | when the attempt started / was submitted, format `day/month/year hour:minute` |
| `Duration` | duration of the attempt, as text such as `11 mins 56 secs` or `1 day 1 hour` |
| `Grade/10.00` | total grade, out of 10 |
| `Q. 1 /0.53` … `Q. 19 /0.53` | score of each of the **19** questions, out of **0.53** (really 10/19 ≈ 0.526, rounded); partial credit is possible |

Note that the **maximum score is written in the column name** (`/10.00`, `/0.53`). We'll read it from there instead of typing `0.53` in our code.

### 4.5 What type does each column have?

In [10]:
for name, values in raw.items():
    print(f"{name:<12} {type(values[0]).__name__}")

Status       str
Started      str
Completed    str
Duration     str
Grade/10.00  str
Q. 1 /0.53   str
Q. 2 /0.53   str
Q. 3 /0.53   str
Q. 4 /0.53   str
Q. 5 /0.53   str
Q. 6 /0.53   str
Q. 7 /0.53   str
Q. 8 /0.53   str
Q. 9 /0.53   str
Q. 10 /0.53  str
Q. 11 /0.53  str
Q. 12 /0.53  str
Q. 13 /0.53  str
Q. 14 /0.53  str
Q. 15 /0.53  str
Q. 16 /0.53  str
Q. 17 /0.53  str
Q. 18 /0.53  str
Q. 19 /0.53  str


As expected from a text file, **everything is a string**. We need to convert:

- `Status` → keep as text
- `Started`, `Completed` → `datetime`
- `Time taken` → a number of **minutes** (`float`)
- `Grade/10.00`, `Q. …` → `float`, with `-` (no answer) → `None`

### 4.6 Keep only the rows we want, explicitly

Removing rows is a **decision** that changes the results, so we make it in its own visible step and report what it did.

In [11]:
def filter_rows(columns, keep):
    """Return a new column dict with only the rows for which keep(row) is True."""
    kept = [i for i in range(num_rows(columns)) if keep(get_row(columns, i))]
    return {name: [values[i] for i in kept] for name, values in columns.items()}


def is_finished(row):
    return row["Status"] == "Finished"


finished = filter_rows(raw, is_finished)
print(f"kept {num_rows(finished)} of {num_rows(raw)} rows "
      f"({num_rows(raw) - num_rows(finished)} removed: in-progress attempts and the average row)")

kept 27 of 28 rows (1 removed: in-progress attempts and the average row)


>**Principle: Don't hide data decisions.** It would be easy to skip these rows quietly inside `load_csv`. But then nobody reading the notebook would know they existed, or that 3 students never submitted. Loading and filtering are **two different jobs**.

## 5. Preprocess: convert each column to a useful type

We write **one small function per kind of value**, test each one on a few examples, and then apply them to the whole table.

### 5.1 Scores

In [12]:
MISSING = "-"   # how Moodle writes "no value"


def parse_score(text):
    """Convert a score such as '0.53' to a float. Return None for '-' (no answer)."""
    text = text.strip()
    if text == MISSING:
        return None
    try:
        return float(text)
    except ValueError:
        raise ValueError(f"cannot parse score {text!r}") from None


assert parse_score("0.53") == 0.53
assert parse_score("10") == 10.0
assert parse_score("-") is None

>**Principle: Defensive code.** A tempting shortcut is
> ```python
> try:
>     col_val = float(col_val)
> except:
>     col_val = 0
> ```
> That turns *every* problem into a score of **0**: a missing answer, a typo, even a bug in our own code. A made-up 0 silently drags every average down. Instead:
> - **known** "no value" (`-`) → `None`, explicitly
> - **anything else** we can't read → an error that names the value
> - never a bare `except:`, it hides every error, including your own typos

### 5.2 Dates and times

Moodle writes dates as **day/month/year** (`8/10/2025` is 8 October, not 10 August).

In [13]:
MOODLE_DATETIME_FORMAT = "%d/%m/%Y %H:%M"


def parse_moodle_datetime(text):
    """Convert '8/10/2025 11:29' (day/month/year) to a datetime. Return None for '-'."""
    text = text.strip()
    if text == MISSING:
        return None
    return dt.datetime.strptime(text, '%m/%d/%Y %H:%M')


assert parse_moodle_datetime("8/20/2025 11:29") == dt.datetime(2025, 8, 20, 11, 29)
assert parse_moodle_datetime("-") is None

### 5.3 Durations

`Time taken` comes in several shapes. Let's look at them **before** writing the parser:

In [14]:
def is_long(text):
    return "hour" in text or "day" in text


print("typical:", [t for t in finished["Duration"] if not is_long(t)][:4])
print("unusual:", [t for t in finished["Duration"] if is_long(t)])

typical: ['15 mins 15 secs', '6 mins 15 secs', '31 mins 58 secs', '14 mins 17 secs']
unusual: ['21 hours 52 mins', '3 hours 37 mins']


Some attempts took **hours**, and some even **days**: the student opened the quiz and submitted it much later. A parser that only knows `hour`, `min` and `sec` would silently read `1 day 1 hour` as 60 minutes!

Every duration is a list of *(amount, unit)* pairs. Instead of an `if/elif` per unit, we put the units in a **table**:

In [15]:
MINUTES_PER_UNIT = {"day": 24 * 60, "hour": 60, "min": 1, "sec": 1 / 60}


def parse_duration(text):
    """Convert a Moodle duration such as '11 mins 56 secs' to minutes (float).

    Return None for '-'. Raise ValueError for an unknown unit or a malformed text.
    """
    text = text.strip()
    if text == MISSING:
        return None
    tokens = text.split()
    if len(tokens) == 0 or len(tokens) % 2 != 0:
        raise ValueError(f"cannot parse duration {text!r}")
    minutes = 0.0
    for amount, unit in zip(tokens[0::2], tokens[1::2]):
        unit = unit.rstrip("s")                       # 'mins' -> 'min', 'days' -> 'day'
        if unit not in MINUTES_PER_UNIT:
            raise ValueError(f"unknown unit {unit!r} in duration {text!r}")
        minutes += int(amount) * MINUTES_PER_UNIT[unit]
    return minutes


assert math.isclose(parse_duration("11 mins 56 secs"), 11 + 56 / 60)
assert parse_duration("21 hours 59 mins") == 21 * 60 + 59
assert parse_duration("1 day 1 hour") == 24 * 60 + 60
assert parse_duration("45 secs") == 0.75
assert parse_duration("-") is None

>**Principle: Data-driven code.** To support a new unit (say `week`), we add **one entry** to `MINUTES_PER_UNIT`; the function doesn't change. Notice also `math.isclose` in the first test: `11 + 56/60` is a float, and floats should not be compared with `==` (why? see Topic 6).

### 5.4 Convert all columns at once

Which parser applies to which column? We answer that question in **one** place:

In [16]:
def keep_text(text):
    return text


def parser_for(column_name):
    """Return the function that converts the values of this column."""
    if column_name in ("Started", "Completed"):
        return parse_moodle_datetime
    if column_name == "Duration":
        return parse_duration
    if "/" in column_name:            # 'Grade/10.00', 'Q. 1 /0.53', ...
        return parse_score
    return keep_text


def convert_columns(columns):
    """Return a new column dict with every column converted by its parser."""
    return {name: [parser_for(name)(value) for value in values]
            for name, values in columns.items()}


quiz = convert_columns(finished)

for name, values in quiz.items():
    print(f"{name:<12} {type(values[0]).__name__}")

Status       str
Started      datetime
Completed    datetime
Duration     float
Grade/10.00  float
Q. 1 /0.53   float
Q. 2 /0.53   float
Q. 3 /0.53   float
Q. 4 /0.53   float
Q. 5 /0.53   float
Q. 6 /0.53   float
Q. 7 /0.53   float
Q. 8 /0.53   float
Q. 9 /0.53   float
Q. 10 /0.53  float
Q. 11 /0.53  float
Q. 12 /0.53  float
Q. 13 /0.53  float
Q. 14 /0.53  float
Q. 15 /0.53  float
Q. 16 /0.53  float
Q. 17 /0.53  float
Q. 18 /0.53  float
Q. 19 /0.53  float


>**Principle: DRY (Don't Repeat Yourself).** A first attempt often puts everything in one big loop, with an if/elif branch per column type and each branch building its own list. Then, to change how scores are parsed, you have to find and fix every branch that does it. Here each piece of knowledge lives in exactly one place: *how* to parse a score (`parse_score`), *which* columns are scores (`parser_for`), and *how* to apply parsers to a table (`convert_columns`). Notice also that `convert_columns` returns a **new** dict: `finished` is unchanged, so re-running a cell never converts data twice.

Finally, some names we'll use a lot. The maximum points are read **from the header**, not typed by hand:

In [17]:
GRADE = "Grade/10.00"
QUESTIONS = [name for name in quiz if name.startswith("Q.")]


def max_points(column_name):
    """'Q. 1 /0.53' -> 0.53 ; 'Grade/10.00' -> 10.0"""
    return float(column_name.split("/")[-1])


print(len(QUESTIONS), "questions,", max_points(QUESTIONS[0]), "points each;",
      "grade out of", max_points(GRADE))

19 questions, 0.53 points each; grade out of 10.0


## 6. Verify: does the converted data make sense?

Converting without errors does **not** mean converting correctly. Before analysing, we check facts that *must* be true if the data and our code are right.

**Check 1.** The grade is the sum of the question scores, rescaled to 10.

Moodle shows each score **rounded to 2 decimals** (a question is really worth 10/19 ≈ 0.5263 points, shown as 0.53). The sum of 19 rounded scores can be off by up to 19 × 0.005 ≈ 0.1, so that is our tolerance.

In [18]:
def check_grades(quiz):
    """Return the rows whose grade differs from the rescaled sum of question scores."""
    total_max = sum(max_points(q) for q in QUESTIONS)
    tolerance = len(QUESTIONS) * 0.005          # rounding to 2 decimals, per question
    problems = []
    for i in range(num_rows(quiz)):
        points = sum(quiz[q][i] for q in QUESTIONS)
        expected = points / total_max * max_points(GRADE)
        if abs(quiz[GRADE][i] - expected) > tolerance:
            problems.append((i, quiz[GRADE][i], round(expected, 2)))
    return problems


assert check_grades(quiz) == [], check_grades(quiz)
print("✅ every grade equals the sum of its question scores")

✅ every grade equals the sum of its question scores


**Check 2.** `Completed` is not before `Started`, and `Duration` matches `Completed − Started`.

Moodle rounds the timestamps to the minute and only shows the two largest units of a duration (`1 day 1 hour` drops the minutes), so we allow a small tolerance.

In [19]:
def check_durations(quiz, parse=parse_duration):
    """Return the rows whose 'Time taken' disagrees with the two timestamps."""
    problems = []
    for i in range(num_rows(quiz)):
        start, end = quiz["Started"][i], quiz["Completed"][i]
        if end < start:
            problems.append((i, "completed before started"))
            continue
        from_timestamps = (end - start).total_seconds() / 60
        reported = parse(finished["Duration"][i])
        tolerance = max(2, 0.05 * from_timestamps)   # 2 min, or 5% for long attempts
        if abs(reported - from_timestamps) > tolerance:
            problems.append((i, finished["Time taken"][i], round(from_timestamps)))
    return problems


assert check_durations(quiz) == [], check_durations(quiz)
print("✅ every 'Time taken' agrees with the timestamps")

✅ every 'Time taken' agrees with the timestamps


>**Principle: Verification.** *"How do I know this is right?"* Cross-check the data against itself: totals that must add up, timestamps that must agree. These checks cost a few lines and catch exactly the kind of bug that never crashes. Write them as `assert`, so a future change that breaks them stops the notebook.

## 7. Explore the converted data

### 7.1 Numeric columns (including dates): missing values, min, max

In [20]:
def is_numeric_column(values):
    return any(isinstance(v, (int, float, dt.datetime)) for v in values)


def show(value):
    """Short display text for a value: floats with 2 decimals."""
    return f"{value:.2f}" if isinstance(value, float) else str(value)


print(f"{'column':<12} {'missing':>7}  {'min':>19}  {'max':>19}")
for name, values in quiz.items():
    if not is_numeric_column(values):
        continue
    present = [v for v in values if v is not None]
    missing = len(values) - len(present)
    print(f"{name:<12} {missing:>7}  {show(min(present)):>19}  {show(max(present)):>19}")

column       missing                  min                  max
Started            0  2026-09-24 14:47:00  2026-09-30 10:00:00
Completed          0  2026-09-24 15:03:00  2026-09-30 10:20:00
Duration           0                 6.25              1312.00
Grade/10.00        0                 3.55                10.00
Q. 1 /0.53         0                 0.00                 0.53
Q. 2 /0.53         0                 0.00                 0.53
Q. 3 /0.53         0                 0.00                 0.53
Q. 4 /0.53         0                 0.00                 0.53
Q. 5 /0.53         0                 0.00                 0.53
Q. 6 /0.53         0                 0.00                 0.53
Q. 7 /0.53         0                 0.00                 0.53
Q. 8 /0.53         0                 0.26                 0.53
Q. 9 /0.53         0                 0.00                 0.53
Q. 10 /0.53        0                 0.13                 0.53
Q. 11 /0.53        0                 0.00              

- No missing values remain (the `-` values were all in the rows we removed).
- Scores are within their limits (0–0.53 per question, 0–10 overall).
- ⚠️ `Duration` ranges from a few minutes to **more than a day**. Let's look closer.

In [21]:
long_attempts = sorted(m for m in quiz["Duration"] if m > 60)
[f"{m / 60:.1f} h" for m in long_attempts]

['3.6 h', '21.9 h']

These students most likely opened the quiz, left, and submitted later. Their `Duration` doesn't measure working time. **Keep this in mind for question 3.**

### 7.2 Text (categorical) columns: distinct values

In [22]:
for name, values in quiz.items():
    if not is_numeric_column(values):
        print(name, Counter(values))

Status Counter({'Finished': 27})


### 7.3 A first look at the grades

In [23]:
grade_counts = Counter(quiz[GRADE])
print("perfect scores (10):", grade_counts[10.0])
print("lowest grades:      ", sorted(quiz[GRADE])[:5])

perfect scores (10): 2
lowest grades:       [3.55, 3.55, 4.61, 5.0, 5.39]


Many students got **10/10**, and one grade is far below all the others. Both are worth investigating in the questions below.

## 8. Refine the questions

Now that we understand the data, we can turn *"How proficient is the class in Python?"* into precise questions that the data can answer:

1. **How are the total grades distributed?**
2. **Which questions did students find hardest?** (lowest average score first)
3. **Is there a relationship between time taken and grade?**
4. **Are there outliers**, students whose grade is unusually low or high?

## 9. Worked example: summary statistics, written and tested by hand

Questions 1, 3 and 4 need the **mean**, the **median** and **percentiles**. We write them ourselves and **test** them against small examples where we know the answer, and against the `statistics` module.

### 9.1 Mean

In [24]:
def mean(values):
    """Arithmetic mean. Raises ValueError for an empty list."""
    if len(values) == 0:
        raise ValueError("mean of an empty list")
    return sum(values) / len(values)


assert mean([1, 2, 3, 4]) == 2.5
assert mean([7]) == 7

### 9.2 Percentiles (and the median)

To find the *p*-th percentile of *n* sorted values, we compute the position $k = \frac{p}{100}(n-1)$.
If $k$ is a whole number, the answer is the value at index $k$. Otherwise, we **interpolate** between the two neighbours.

**Example:** `[1, 2, 3, 4]`, median (*p* = 50) → $k = 1.5$ → halfway between index 1 (`2`) and index 2 (`3`) → **2.5**.

The first version of this notebook used `sorted_grades[int(k)]`, which always rounds **down**: for `[1, 2, 3, 4]` it returns 2 instead of 2.5. With 66 students (an even number), our median would have been wrong.

In [25]:
def percentile(values, p):
    """p-th percentile (0 <= p <= 100) with linear interpolation between neighbours."""
    if len(values) == 0:
        raise ValueError("percentile of an empty list")
    if not 0 <= p <= 100:
        raise ValueError(f"p must be between 0 and 100, got {p}")
    ordered = sorted(values)
    k = p / 100 * (len(ordered) - 1)
    lower = math.floor(k)
    upper = math.ceil(k)
    fraction = k - lower
    return ordered[lower] + fraction * (ordered[upper] - ordered[lower])


def median(values):
    return percentile(values, 50)


# Tests on lists where we know the answer
assert median([3, 1, 2]) == 2
assert median([4, 1, 3, 2]) == 2.5
assert percentile([10, 20, 30, 40, 50], 0) == 10
assert percentile([10, 20, 30, 40, 50], 100) == 50
assert percentile([10, 20, 30, 40, 50], 25) == 20

# Tests against the standard library, on the real data
grades = quiz[GRADE]
assert math.isclose(median(grades), statistics.median(grades))
q1, q2, q3 = statistics.quantiles(grades, n=4, method="inclusive")
assert math.isclose(percentile(grades, 25), q1)
assert math.isclose(percentile(grades, 75), q3)
print("✅ mean, median and percentile pass all tests")

✅ mean, median and percentile pass all tests


In [26]:
print(f"mean grade:   {mean(grades):.2f}")
print(f"median grade: {median(grades):.2f}")

mean grade:   7.31
median grade: 7.63


>**Principle: Testing.** A test is a check you write down so the computer repeats it every time. Good tests use **small inputs where you know the answer** (`[4, 1, 3, 2]` → 2.5), cover **edge cases** (*p* = 0, *p* = 100, one value), and, when available, compare with a **trusted implementation**. If you change `percentile` later and break it, the `assert` stops the notebook immediately.

## 10. Exercises (homework)

For each question:
1. Complete the functions marked `# TODO`. Replace `raise NotImplementedError` with your code.
2. Run the **test cell** below them. All `assert`s must pass.
3. Apply your functions to the quiz data.
4. Answer the ❓ **Explain why** questions in the markdown cells.

Use the functions from sections 3–9 (`mean`, `percentile`, `median`, `filter_rows`, …). Don't copy their code.

### Question 1: How are the total grades distributed?

The median gives one number. To *see* the distribution, we split the range [0, 10] into **bins** and count the students in each bin:
`[0, 2)`, `[2, 4)`, `[4, 6)`, `[6, 8)`, `[8, 10]`. The last bin includes 10.

We store the counts in a dictionary: the **key** is a tuple `(low, high)` with the two ends of a bin, the **value** is the number of students.

In [27]:
def make_bins(start, stop, width):
    """Return equal-width bins as tuples: make_bins(0, 10, 2) -> [(0, 2), (2, 4), ..., (8, 10)]."""
    lst = []
    current = start
    while current < stop:
        next_bin = min(current + width, stop)
        lst.append((current, next_bin))
        current = next_bin
    return lst


def count_in_bins(values, bins):
    """Return {bin: number of values in the bin}.

    Every bin is [low, high), except the last one, which is [low, high].
    Raise ValueError if a value falls outside all bins.
    """
    for value in values:
        if not any(low <= value < high for low, high in bins[:-1]) and not (bins[-1][0] <= value <= bins[-1][1]):
            raise ValueError(f"value {value} falls outside all bins")
    counts = {bin: 0 for bin in bins}
    for value in values:
        for bin in bins:
            low, high = bin
            if (low <= value < high) or (bin == bins[-1] and low <= value <= high):
                counts[bin] += 1
                break
    return counts


def text_histogram(counts):
    """Return a text histogram, one line per bin, e.g. '[0, 2)  ##### 5'."""
    lines = []
    bins = sorted(counts)

    for i, bin in enumerate(bins):
        low, high = bin
        count = counts[bin]

        closing = "]" if i == len(bins) - 1 else ")"
        bin_str = f"[{low}, {high}{closing}"

        if count == 0:
            lines.append(f"{bin_str}   0")
        else:
            lines.append(f"{bin_str}   {'#' * count} {count}")

    return "\n".join(lines)
    

In [28]:
# Tests for Question 1: do not change
assert make_bins(0, 10, 2) == [(0, 2), (2, 4), (4, 6), (6, 8), (8, 10)]
assert make_bins(0, 10, 5) == [(0, 5), (5, 10)]

bins = [(0, 5), (5, 10)]
assert count_in_bins([0, 1.9, 4.99, 5, 9.99, 10], bins) == {(0, 5): 3, (5, 10): 3}
assert count_in_bins([], bins) == {(0, 5): 0, (5, 10): 0}
try:
    count_in_bins([11], bins)
    assert False, "count_in_bins should raise ValueError for 11"
except ValueError:
    pass

assert text_histogram({(0, 5): 3, (5, 10): 0}).splitlines() == ["[0, 5)   ### 3", "[5, 10]   0"]
print("✅ Question 1: all tests pass")

✅ Question 1: all tests pass


In [29]:
# Apply to the quiz data: print the histogram of the grades with 5 bins
bins = make_bins(0, 10, 2)
counts = count_in_bins(quiz[GRADE], bins)

print(text_histogram(counts))

[0, 2)   0
[2, 4)   ## 2
[4, 6)   ###### 6
[6, 8)   ######## 8
[8, 10]   ########### 11


❓ **Explain why**

- **1a.** Could the key be a **list** `[low, high]` instead of a tuple `(low, high)`? Try it, read the error message, and explain.
- **1b.** Describe the shape of the distribution in one or two sentences. Is the mean or the median a better summary here? Why?
- **1c.** 2 students scored a perfect 10. What does that tell you about the quiz itself (not only about the students)?

**1a.** Không thể sử dụng list làm key của dictionary vì list là kiểu dữ liệu có thể thay đổi (mutable) và không thể hash (unhashable). Trong khi đó, tuple chứa các giá trị số là kiểu dữ liệu bất biến (immutable) và có thể hash (hashable), nên có thể dùng làm key của dictionary.

**1b.** Phân phối điểm có xu hướng **lệch trái** vì phần lớn sinh viên đạt điểm cao, trong khi số sinh viên có điểm thấp tương đối ít. Cụ thể, có 11/27 sinh viên đạt từ 8 đến 10 điểm, 8 sinh viên đạt từ 6 đến dưới 8 điểm và không có sinh viên nào dưới 2 điểm. **trung vị** phù hợp hơn để mô tả mức điểm điển hình vì ít bị ảnh hưởng bởi các giá trị điểm thấp, trong khi Mean (trung bình cộng) có thể bị kéo xuống bởi các giá trị này.

**1c.** Trong tổng số 27 sinh viên, chỉ có 2 sinh viên đạt điểm tuyệt đối 10. Điều này cho thấy bài quiz không quá dễ để tất cả sinh viên đều đạt điểm tối đa và vẫn có khả năng phân biệt mức độ thành thạo giữa các sinh viên. Tuy nhiên, phần lớn sinh viên đạt điểm tương đối cao, cho thấy bài quiz có thể phù hợp với kiến thức đã học hoặc có độ khó vừa phải. Cần xem xét thêm nội dung và khả năng phân hóa của từng câu hỏi để đánh giá chính xác chất lượng bài kiểm tra.

### Question 2: Which questions did students find hardest?

Compute the **average score of each question**, and sort the questions from lowest average (hardest) to highest (easiest). Return a dictionary `{question name: average}` in that order.

In [30]:
def question_averages(quiz, questions):
    """Return {question: average score}, sorted from the lowest to the highest average."""
    averages = {q: mean([score for score in quiz[q] if score is not None]) for q in questions}
    return dict(sorted(averages.items(), key=lambda item: item[1]))

In [31]:
# Tests for Question 2: do not change
tiny = {"Q. 1 /1": [1.0, 1.0], "Q. 2 /1": [0.0, 0.5], "Q. 3 /1": [0.0, 1.0]}
result = question_averages(tiny, ["Q. 1 /1", "Q. 2 /1", "Q. 3 /1"])
assert list(result) == ["Q. 2 /1", "Q. 3 /1", "Q. 1 /1"]
assert result["Q. 2 /1"] == 0.25
print("✅ Question 2: all tests pass")

✅ Question 2: all tests pass


In [32]:
# Apply to the quiz data: show the 5 hardest questions, with their average as a % of the maximum
averages = question_averages(quiz, QUESTIONS)
for q, avg in list(averages.items())[:5]:
    print(f"{q:<12} {avg / max_points(q) * 100:.1f}%")

Q. 5 /0.53   51.9%
Q. 19 /0.53  55.6%
Q. 13 /0.53  59.3%
Q. 14 /0.53  59.3%
Q. 17 /0.53  59.3%


❓ **Explain why**

- **2a.** Why is it useful to show the average as a **percentage** of the maximum, rather than the raw average?
- **2b.** Some questions give **partial credit** (e.g. 0.26 out of 0.53). Two questions can have the same average but very different stories. Give an example, and suggest a second statistic that would tell them apart.
- **2c.** As the lecturer, what would you do with the result of this question?

**2a.** Việc sử dụng phần trăm điểm tối đa giúp so sánh mức độ khó giữa các câu hỏi một cách công bằng hơn, đặc biệt khi các câu có điểm tối đa khác nhau. Nếu chỉ sử dụng điểm trung bình thô, những câu có điểm tối đa thấp có thể bị đánh giá nhầm là khó hơn.

**2b.** Hai câu hỏi có thể có cùng điểm trung bình nhưng phân phối điểm khác nhau. Ví dụ, câu A có điểm [0, 0, 1, 1], còn câu B có điểm [0.5, 0.5, 0.5, 0.5]. Cả hai đều có điểm trung bình là 0.5, nhưng câu A có sự phân hóa rõ rệt giữa các sinh viên, trong khi câu B có mức điểm đồng đều. Có thể sử dụng **độ lệch chuẩn** để phân biệt hai trường hợp này.

**2c.** Dựa trên kết quả phân tích, câu 5 là câu khó nhất với tỷ lệ điểm trung bình 51.9%, tiếp theo là câu 19 với 55.6% và các câu 13, 14, 17 với khoảng 59.3%. Nếu là giảng viên, em sẽ xem xét lại nội dung của những câu hỏi này để xác định nguyên nhân sinh viên đạt điểm thấp, đồng thời bổ sung giải thích và bài tập thực hành cho các kiến thức liên quan. Em cũng sẽ kiểm tra cách diễn đạt và độ phù hợp của câu hỏi nhằm cải thiện chất lượng bài quiz.

### Question 3: Is there a relationship between Duration and grade?

**Goal:** find out whether students who worked faster or slower got different grades.

#### Step 0: a data decision (required)

In section 7.1 we found attempts that lasted **hours or days**. Their `Duration` doesn't measure working time.

❓ **3a.** Decide what to do with these attempts (keep them, remove them, or cap them at some value), and **justify** your decision. If you remove them, choose a threshold, and report how many attempts you removed.

**3a.** Em chọn loại bỏ các lượt làm bài có thời gian vượt quá 60 phút vì những trường hợp kéo dài nhiều giờ hoặc nhiều ngày có thể xuất phát từ việc sinh viên mở bài kiểm tra nhưng không làm liên tục. Do đó, thời gian được ghi nhận không nhất thiết phản ánh thời gian thực sự dành cho bài quiz. Việc loại bỏ những trường hợp này giúp so sánh điểm số giữa các nhóm làm nhanh và chậm hợp lý hơn. Tuy nhiên, ngưỡng 60 phút chỉ là giả định phân tích và có thể loại bỏ cả những lượt làm bài hợp lệ.

In [33]:
MAX_MINUTES = 60

timed = filter_rows(
    quiz,
    lambda row: row["Duration"] <= MAX_MINUTES
)

print(f"Kept: {num_rows(timed)} attempts")
print(f"Removed: {num_rows(quiz) - num_rows(timed)} attempts")

Kept: 25 attempts
Removed: 2 attempts


#### Simple version: fast group vs. slow group

1. Compute the **mean** and **median** time taken.
2. Split the students into a **fast** group (time < median) and a **slow** group (time ≥ median).
3. Compare the mean grade of the two groups.

Possible outcomes:
- **The fast group scores higher:** strong students may know the answers and finish quickly.
- **The slow group scores higher:** careful students may get better results by taking their time.
- **Similar grades:** no clear relationship between time and grade.

In [34]:
def mean_by_group(values, groups):
    """Return {group: mean of the values in that group}.

    values and groups are lists of the same length; groups[i] is the group of values[i].
    """
    group_values = {}
    for value, group in zip(values, groups):
        if group not in group_values:
            group_values[group] = []
        group_values[group].append(value)
    return {group: mean(vals) for group, vals in group_values.items()}

In [35]:
# Tests for Question 3: do not change
assert mean_by_group([1, 2, 3, 4], ["a", "b", "a", "b"]) == {"a": 2, "b": 3}
assert mean_by_group([5], ["x"]) == {"x": 5}
print("✅ Question 3: all tests pass")

✅ Question 3: all tests pass


In [36]:
# Apply: mean & median time, fast/slow groups, mean grade per group
times = timed["Duration"]

mean_time = mean(times)
median_time = median(times)

print(f"Mean time: {mean_time:.2f} minutes")
print(f"Median time: {median_time:.2f} minutes")
groups = [
    "Fast" if t < median_time else "Slow"
    for t in times
]

grade_by_group = mean_by_group(timed[GRADE], groups)

print("Mean grades by group:", grade_by_group)

Mean time: 19.84 minutes
Median time: 19.40 minutes
Mean grades by group: {'Fast': 6.9075, 'Slow': 7.9253846153846155}


#### Advanced version (still no libraries)

- Split the times into intervals (e.g. 0–10 min, 10–20 min, 20–30 min, …). You can reuse `make_bins` and adapt `count_in_bins`.
- Compute the mean grade in each interval, and the number of students in it.
- Does the grade go up or down with time?

In [37]:
bins = make_bins(0, 60, 10)

for i, (low, high) in enumerate(bins):
    selected = [
        j for j, t in enumerate(timed["Duration"])
        if low <= t < high or (i == len(bins) - 1 and t == high)
    ]

    if selected:
        scores = [timed[GRADE][j] for j in selected]
        print(f"[{low}, {high}]: n={len(selected)}, mean={mean(scores):.2f}")

[0, 10]: n=3, mean=6.71
[10, 20]: n=11, mean=6.85
[20, 30]: n=9, mean=8.13
[30, 40]: n=1, mean=9.61
[40, 50]: n=1, mean=7.63


❓ **Explain why**

- **3b.** Describe what you found in one or two sentences, and include the number of attempts you removed.
- **3c.** We split the groups at the **median** time, not at the mean. Why?
- **3d.** Suppose the slow group has clearly higher grades. Can we conclude that "taking more time *causes* better grades"? Give another possible explanation.

**3b.** Sau khi loại bỏ 2 lượt làm bài vượt quá 60 phút, còn lại 25 lượt để phân tích. Thời gian làm bài trung bình là **19.84 phút**, trung vị là **19.40 phút**. Nhóm **Fast** có điểm trung bình 6.91, thấp hơn nhóm **Slow** với 7.93 điểm. Kết quả cho thấy nhóm làm bài chậm có xu hướng đạt điểm cao hơn khoảng 1.02 điểm. Tuy nhiên, điều này chỉ phản ánh mối liên hệ trong dữ liệu hiện tại, chưa chứng minh quan hệ nhân quả.

**3c.** Sử dụng **median (trung vị)** thay vì mean (trung bình cộng) để chia nhóm vì median ít bị ảnh hưởng bởi các giá trị thời gian bất thường. Ngoài ra, median giúp chia sinh viên thành hai nhóm có số lượng tương đối cân bằng, từ đó việc so sánh điểm trung bình giữa nhóm Fast và Slow hợp lý hơn.

**3d.** Không thể kết luận rằng dành nhiều thời gian làm bài hơn sẽ trực tiếp giúp sinh viên đạt điểm cao hơn, vì **tương quan không đồng nghĩa với quan hệ nhân quả**. Những yếu tố khác như kiến thức nền, mức độ chuẩn bị, sự cẩn thận và thói quen kiểm tra lại đáp án có thể ảnh hưởng đến cả thời gian làm bài lẫn điểm số. Để xác định quan hệ nhân quả cần có phương pháp nghiên cứu phù hợp và kiểm soát các yếu tố gây nhiễu.

### Question 4: Are there outliers?

**Goal:** find students whose grade is unusually low or high compared with the rest of the class.

#### Method 1: interquartile range (IQR)

1. **Q1** = 25th percentile, **Q3** = 75th percentile, **IQR** = Q3 − Q1
2. **Lower fence** = Q1 − 1.5 × IQR, **upper fence** = Q3 + 1.5 × IQR
3. A grade below the lower fence is a **low outlier**; above the upper fence, a **high outlier**.

*Example:* Q1 = 4.5, Q3 = 7.5 → IQR = 3.0 → fences 0 and 12 (the maximum grade is 10, so no high outlier is possible).

#### Method 2: standard deviation

1. Compute the mean and the **standard deviation** $\sigma = \sqrt{\frac{1}{n}\sum_i (x_i - \bar{x})^2}$ (the *population* standard deviation).
2. **Lower fence** = mean − 2σ, **upper fence** = mean + 2σ.

*Example:* mean = 6.0, σ = 1.5 → fences 3.0 and 9.0.

Both functions return a tuple `(lower fence, upper fence, outliers)`, where `outliers` is the list of values outside the fences.

In [38]:
def iqr_outliers(values, k=1.5):
    """Return (lower fence, upper fence, list of outliers) using the IQR rule."""
    q1 = percentile(values, 25)
    q3 = percentile(values, 75)

    iqr = q3 - q1

    low = q1 - k * iqr
    high = q3 + k * iqr

    outliers = [x for x in values if x < low or x > high]

    return low, high, outliers


def std_dev(values):
    """Population standard deviation (divide by n)."""
    avg = mean(values)

    variance = sum((x - avg) ** 2 for x in values) / len(values)

    return math.sqrt(variance)


def std_outliers(values, k=2):
    """Return (lower fence, upper fence, list of outliers) using mean ± k standard deviations."""
    avg = mean(values)
    std = std_dev(values)

    low = avg - k * std
    high = avg + k * std

    outliers = [x for x in values if x < low or x > high]

    return low, high, outliers

In [39]:
# Tests for Question 4: do not change
data = [2, 4, 4, 4, 5, 5, 7, 9]
assert std_dev(data) == 2.0
assert math.isclose(std_dev(quiz[GRADE]), statistics.pstdev(quiz[GRADE]))

assert iqr_outliers([1, 2, 3, 4, 5, 6, 7, 8, 100]) == (-3.0, 13.0, [100])
assert iqr_outliers([1, 2, 3]) == (0.0, 4.0, [])

low, high, outliers = std_outliers(data)
assert (low, high, outliers) == (1.0, 9.0, [])
print("✅ Question 4: all tests pass")

✅ Question 4: all tests pass


In [40]:
# Apply both methods to the grades and print the fences and the outliers
grades = quiz[GRADE]

iqr_low, iqr_high, iqr_values = iqr_outliers(grades)
std_low, std_high, std_values = std_outliers(grades)

print("IQR Method:")
print(f"Lower fence: {iqr_low:.2f}")
print(f"Upper fence: {iqr_high:.2f}")
print(f"Outliers: {iqr_values}")

print()

print("Standard Deviation Method:")
print(f"Lower fence: {std_low:.2f}")
print(f"Upper fence: {std_high:.2f}")
print(f"Outliers: {std_values}")

IQR Method:
Lower fence: 2.17
Upper fence: 12.17
Outliers: []

Standard Deviation Method:
Lower fence: 3.72
Upper fence: 10.91
Outliers: [3.55, 3.55]


❓ **Explain why**

- **4a.** Do the two methods find the same outliers? Explain any difference. (Hint: what does one very low grade do to the mean and to σ? And to Q1 and Q3?)
- **4b.** The `statistics` module has both `pstdev` (divide by *n*) and `stdev` (divide by *n − 1*). When would you use each?
- **4c.** You found an outlier. Should you remove it from the data? What would you do instead?

**4a.** Hai phương pháp cho kết quả khác nhau. Phương pháp **IQR** có cận dưới là 2.17 và cận trên là 12.17 nên không phát hiện giá trị ngoại lệ nào. Trong khi đó, phương pháp **Standard Deviation** có cận dưới là 3.72 và cận trên là 10.91, phát hiện hai giá trị ngoại lệ đều bằng 3.55. Sự khác biệt xuất phát từ cách xác định ngưỡng: IQR dựa trên các tứ phân vị nên ít bị ảnh hưởng bởi giá trị cực đoan, còn Standard Deviation dựa trên trung bình và độ lệch chuẩn. Vì vậy, hai phương pháp có thể đánh giá cùng một giá trị theo những cách khác nhau.

**4b.** Sử dụng `statistics.pstdev()` khi dữ liệu được xem là toàn bộ tổng thể cần phân tích, với phương sai được tính bằng cách chia cho $n$. Ngược lại, `statistics.stdev()` được sử dụng khi dữ liệu là một mẫu đại diện cho tổng thể lớn hơn, với phương sai được tính bằng cách chia cho $n-1$.

**4c.** Không nên tự động loại bỏ các giá trị ngoại lệ vì chúng có thể phản ánh kết quả thực tế chứ không phải lỗi dữ liệu. Trong trường hợp này, hai giá trị 3.55 có thể là điểm hợp lệ của hai sinh viên. Cần kiểm tra nguồn dữ liệu và nguyên nhân xuất hiện các giá trị này trước khi quyết định xử lý. Nếu dữ liệu hợp lệ, nên giữ lại và có thể so sánh kết quả phân tích khi có và không có ngoại lệ để đánh giá mức độ ảnh hưởng.

✅ **Before you submit:** *Kernel → Restart & Run All*. Every test must pass, and every ❓ must have an answer.